In [ ]:
import pandas as pd
import joblib

# Load the saved model
model_path = "runtime_rf_model.joblib"
loaded_model = joblib.load(model_path)
print("Model loaded from:", model_path)

# Example: predict for a new circuit
example_circuit = {
    "n_qubits": 25,
    "depth": 20,
    "shot": 3000,
    "twoq_total": 320
}

example_df = pd.DataFrame([example_circuit])
predicted_runtime = loaded_model.predict(example_df)[0]
print(f"Predicted run_time for example circuit: {predicted_runtime:.6f} seconds")

In [ ]:
import math, random, time, json, os
import pandas as pd
import joblib
import cudaq

# -----------------------------
# 0) Reuse your circuit builder
# -----------------------------
def build_random_circuit(n_qubits: int,
                         depth: int,
                         shots: int,
                         *,
                         seed: int | None = None,
                         twoq_density: float = 0.5,
                         entangler: str = "cx",
                         ring_connectivity: bool = True,
                         measure: bool = True):
    rng = random.Random(seed)
    k = cudaq.make_kernel()
    q = k.qalloc(n_qubits)

    stats = {
        "n_qubits": n_qubits,
        "depth": depth,
        "twoq_density": twoq_density,
        "entangler": entangler,
        "ring_connectivity": ring_connectivity,
        "measure": measure,
        "shot": shots,
        "cx": 0,
        "cz": 0,
        "twoq_total": 0,
    }

    def rand_u1(target):
        r = rng.random()
        if r < 0.25:
            k.h(target)
        elif r < 0.50:
            k.rx(2 * math.pi * rng.random(), target)
        elif r < 0.75:
            k.ry(2 * math.pi * rng.random(), target)
        else:
            k.rz(2 * math.pi * rng.random(), target)

    for _ in range(depth):
        for i in range(n_qubits):
            rand_u1(q[i])

        if ring_connectivity:
            pairs = [(i, (i + 1) % n_qubits) for i in range(n_qubits)]
        else:
            pairs = [(i, i + 1) for i in range(0, n_qubits - 1, 2)]

        for a, b in pairs:
            if a == b:
                continue
            if rng.random() <= twoq_density:
                if entangler == "cx":
                    k.cx(q[a], q[b])
                    stats["cx"] += 1
                else:
                    k.cz(q[a], q[b])
                    stats["cz"] += 1
                stats["twoq_total"] += 1

    if measure:
        for i in range(n_qubits):
            k.mz(q[i])

    return k, stats

# -----------------------------
# 1) Load trained model
# -----------------------------
model_path = "runtime_rf_model.joblib"
model = joblib.load(model_path)

feature_cols = ["n_qubits", "depth", "shot", "twoq_total"]

# -----------------------------
# 2) Define 5 test circuits
# -----------------------------
# Generate 50 randomized test_specs
# -----------------------------
rng = random.Random(42)   # fixed seed for reproducibility

test_specs = []

base_seed = 101

for i in range(50):
    n = rng.randint(2, 29)
    depth = rng.randint(2, 25)
    shots = rng.choice([1000, 2000, 3000])
    twoq_density = rng.choice([0.3, 0.4, 0.5, 0.6, 0.7])
    entnglr = rng.choice(["cx", "cz"])
    seed = base_seed + i

    test_specs.append(
        (n, depth, shots, twoq_density, entnglr, seed)
    )

# Optional: quick check
print("Generated test_specs (first 5 shown):")
for spec in test_specs[:5]:
    print(spec)

print(f"\nTotal circuits generated: {len(test_specs)}")

# NOTE:
# We do NOT force twoq_total directly.
# Instead, we build the circuit and extract the resulting twoq_total
# to feed the model exactly as in your training pipeline.

# -----------------------------
# 3) Execute + compare
# -----------------------------
rows = []

# Optional: warm-up run (helps reduce first-run compilation overhead variability)
warm_k, _ = build_random_circuit(4, 2, 1000, seed=999, twoq_density=0.5, entangler="cx")
_ = cudaq.sample(warm_k, shots_count=1000)

for (n, depth, shots, twoq_density, entangler, seed) in test_specs:
    kernel, stats = build_random_circuit(
        n, depth, shots,
        seed=seed,
        twoq_density=twoq_density,
        entangler=entangler,
        ring_connectivity=True,
        measure=True
    )

    # Predict using extracted features
    feat = {
        "n_qubits": stats["n_qubits"],
        "depth": stats["depth"],
        "shot": stats["shot"],
        "twoq_total": stats["twoq_total"],
    }
    feat_df = pd.DataFrame([feat])
    pred = float(model.predict(feat_df)[0])

    # Measure actual CUDA-Q runtime (wall clock around cudaq.sample)
    t0 = time.time()
    _ = cudaq.sample(kernel, shots_count=shots)
    t1 = time.time()
    actual = float(t1 - t0)

    abs_err = abs(actual - pred)
    rel_err = abs_err / actual if actual > 0 else float("nan")

    row = {
        **feat,
        "twoq_density": twoq_density,
        "entangler": entangler,
        "seed": seed,
        "pred_run_time_s": pred,
        "actual_run_time_s": actual,
        "abs_err_s": abs_err,
        "rel_err": rel_err,
    }
    rows.append(row)

df_val = pd.DataFrame(rows)

# -----------------------------
# 4) Print + save results
# -----------------------------
pd.set_option("display.max_columns", None)
print(df_val.sort_values("actual_run_time_s"))

print("\nAggregate metrics (n=5):")
print("Mean abs err (s):   ", df_val["abs_err_s"].mean())
print("Median abs err (s): ", df_val["abs_err_s"].median())
print("Mean rel err:       ", df_val["rel_err"].mean())
print("Median rel err:     ", df_val["rel_err"].median())

out_csv = "model_validation_pred_vs_actual.csv"
df_val.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

In [ ]:
import json
import os
import random

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

# -----------------------------
# Config
# -----------------------------
json_glob_indices = range(1, 21)   # 1..20
model_path = "runtime_rf_model.joblib"

feature_cols = ["n_qubits", "depth", "shot", "twoq_total"]
target_col = "run_time"

rng = random.Random(42)

# -----------------------------
# 1) Load all JSON files -> DataFrame
# -----------------------------
all_records = []
missing = []

for i in json_glob_indices:
    fn = f"rand_circuits_batch_{i}.json"
    if not os.path.exists(fn):
        missing.append(fn)
        continue
    with open(fn, "r") as f:
        batch = json.load(f)  # list[dict]
        all_records.extend(batch)

if len(all_records) == 0:
    raise RuntimeError("No benchmark records loaded. Check JSON filenames/paths.")

df = pd.DataFrame(all_records)

# Basic column checks
needed = set(feature_cols + [target_col])
missing_cols = needed - set(df.columns)
if missing_cols:
    raise ValueError(f"Missing columns in JSON data: {sorted(missing_cols)}")

# Ensure numeric types (defensive)
for c in feature_cols + [target_col]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)

print(f"Loaded {len(df):,} circuits from {len(json_glob_indices)} files.")
if missing:
    print("Warning: missing files:")
    for fn in missing:
        print(" -", fn)

# -----------------------------
# 2) Load trained model
# -----------------------------
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Trained model not found: {model_path}")

model = joblib.load(model_path)

# -----------------------------
# 3) Sample 20 real circuits from collected data
# -----------------------------
sampled_df = df.sample(n=20, random_state=42).copy()

X_sampled = sampled_df[feature_cols]
sampled_df["pred_run_time"] = model.predict(X_sampled)

# Optional: quick numeric sanity check on sampled circuits
abs_err = np.abs(sampled_df["pred_run_time"] - sampled_df["run_time"])
print(f"Sampled-20 | mean abs error: {abs_err.mean():.6f} s")
print(f"Sampled-20 | median abs error: {np.median(abs_err):.6f} s")

# -----------------------------
# 4) Create 5 feature-only synthetic circuits + predict
#    (sample feature ranges from dataset to stay realistic)
# -----------------------------
def make_synthetic_samples(df_ref: pd.DataFrame, k: int = 5, seed: int = 7) -> pd.DataFrame:
    r = np.random.default_rng(seed)

    # Use empirical ranges; for shots, keep it to observed discrete set if desired
    n_min, n_max = int(df_ref["n_qubits"].min()), int(df_ref["n_qubits"].max())
    d_min, d_max = int(df_ref["depth"].min()), int(df_ref["depth"].max())
    twoq_min, twoq_max = int(df_ref["twoq_total"].min()), int(df_ref["twoq_total"].max())

    # Shots often come from a small set in your generator; infer common unique values
    shot_values = sorted(df_ref["shot"].dropna().unique().astype(int).tolist())
    if len(shot_values) == 0:
        shot_values = [1000, 2000, 3000]

    rows = []
    for _ in range(k):
        n = int(r.integers(n_min, n_max + 1))
        depth = int(r.integers(d_min, d_max + 1))
        shot = int(r.choice(shot_values))
        twoq = int(r.integers(twoq_min, twoq_max + 1))

        rows.append({"n_qubits": n, "depth": depth, "shot": shot, "twoq_total": twoq})

    return pd.DataFrame(rows)

synthetic_df = make_synthetic_samples(df, k=5, seed=7)
synthetic_df["pred_run_time"] = model.predict(synthetic_df[feature_cols])

print("\nSynthetic samples (feature-only) with predicted runtime:")
print(synthetic_df)

# -----------------------------
# 5) Visualization: overlay on full dataset
#    - Background: full data (measured run_time)
#    - Sampled 20: measured + predicted
#    - Synthetic 5: predicted
# -----------------------------
plots = [
    ("n_qubits", "Number of Qubits"),
    ("depth", "Circuit Depth"),
    ("shot", "Shots"),
    ("twoq_total", "Total Two-Qubit Gates"),
]

for xcol, xlabel in plots:
    plt.figure()

    # Background distribution (full dataset, measured)
    plt.scatter(df[xcol], df["run_time"], alpha=0.15, s=10, label="All benchmarked (measured)")

    # Overlay: sampled 20 (measured)
    plt.scatter(sampled_df[xcol], sampled_df["run_time"], s=60, marker="o", label="Sampled-20 (measured)")

    # Overlay: sampled 20 (predicted)
    plt.scatter(sampled_df[xcol], sampled_df["pred_run_time"], s=60, marker="x", label="Sampled-20 (predicted)")

    # Overlay: synthetic 5 (predicted only)
    plt.scatter(synthetic_df[xcol], synthetic_df["pred_run_time"], s=90, marker="^", label="Synthetic-5 (predicted)")

    plt.xlabel(xlabel)
    plt.ylabel("Runtime (seconds)")
    plt.title(f"{xcol} vs runtime (measured corpus with predicted overlays)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

# -----------------------------
# 6) (Recommended) Extra plot: Predicted vs Measured for sampled-20
# -----------------------------
plt.figure()
plt.scatter(sampled_df["run_time"], sampled_df["pred_run_time"], s=70)
minv = min(sampled_df["run_time"].min(), sampled_df["pred_run_time"].min())
maxv = max(sampled_df["run_time"].max(), sampled_df["pred_run_time"].max())
plt.plot([minv, maxv], [minv, maxv])  # y=x reference
plt.xlabel("Measured runtime (s)")
plt.ylabel("Predicted runtime (s)")
plt.title("Sampled-20: predicted vs measured runtime")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Load the second column (elapsed time), skip header line
data = np.loadtxt("saturn-data/elapsed-time.txt", skiprows=1)

# Column 0 = batch number, Column 1 = elapsed time
elapsed_times = data[:, 1]

# Calculate total and average
total_time = np.sum(elapsed_times)
average_time = np.mean(elapsed_times)

print(f"Total elapsed time: {total_time:.4f}")
print(f"Average elapsed time: {average_time:.4f}")

In [ ]:
print(f"lambda server")

# Load the second column (elapsed time), skip header line
data = np.loadtxt("lambda-data/elapsed-time.txt", skiprows=1)
elapsed_times = data[:, 1]

# Calculate total and average
total_time = np.sum(elapsed_times)
average_time = np.mean(elapsed_times)

print(f"Total elapsed time: {total_time:.4f}")
print(f"Average elapsed time: {average_time:.4f}")

In [ ]:
print(f"reactor server")

# Load the second column (elapsed time), skip header line
data = np.loadtxt("reactor-data/elapsed-time.txt", skiprows=1)
elapsed_times = data[:, 1]

# Calculate total and average
total_time = np.sum(elapsed_times)
average_time = np.mean(elapsed_times)

print(f"Total elapsed time: {total_time:.4f}")
print(f"Average elapsed time: {average_time:.4f}")

In [ ]:
print(f"typhoon server")

# Load the second column (elapsed time), skip header line
data = np.loadtxt("typhon-data/elapsed-time.txt", skiprows=1)
elapsed_times = data[:, 1]

# Calculate total and average
total_time = np.sum(elapsed_times)
average_time = np.mean(elapsed_times)

print(f"Total elapsed time: {total_time:.4f}")
print(f"Average elapsed time: {average_time:.4f}")

In [ ]:
print(f"cardinal server")

# Load the second column (elapsed time), skip header line
data = np.loadtxt("cardinal-data/elapsed-time.txt", skiprows=1)
elapsed_times = data[:, 1]

# Calculate total and average
total_time = np.sum(elapsed_times)
average_time = np.mean(elapsed_times)

print(f"Total elapsed time: {total_time:.4f}")
print(f"Average elapsed time: {average_time:.4f}")